# Notebook 22e — Wu 2003 S-B3: large-scale seedable data generation (widened lognormal prior)

## Purpose

Part of the 2026-09-25 large-scale SBC campaign: generate 20,000-50,000 training draws for the
S-B3 problem (`[alpha, beta_r, eta_col, xi_reb]` under S-B instrumentation) split across ~4-6
machines, using the widened truncated-lognormal prior (`cstr_sbi.recycle.sb3.LOGN_SIGMA_SB3_4D_WIDE`)
that replaces `notebooks/22b`'s narrow-lognormal `eta_col`/`xi_reb` marginals. A 20-seed sweep on
the narrow prior (`notebooks/31c`, `results/31c_extended_seed_sweep_confirm_sbc.csv`) found 0/20
seeds pass full SBC, with `eta_col`/`xi_reb` the persistent worst offenders (KS p as low as
`1e-32`) — the narrow prior puts the worst named fault value (`eta_col=0.80`, `xi_reb=0.70`) only
~2.2 log-sigma into the tail. The widened prior puts it at exactly 1 log-sigma instead
(`sigma_eta_col=0.2231`, `sigma_xi_reb=0.3567`), while keeping `alpha`/`beta_r` exactly as before
(Uniform, already validated) and keeping `eta_col`/`xi_reb` in the *lognormal* family — not a
one-sided half-normal, which was considered and rejected this session (see `sb3.py`'s
`LOGN_SIGMA_SB3_4D_WIDE` docstring: a one-sided cut would put `xi_reb`'s common healthy truth,
`xi_reb=1.0`, exactly on the support's edge, since `xi_reb`'s box `[0.40,1.20]` has nominal in its
*interior*, unlike `eta_col`'s box which already caps at 1.0).

## This notebook is designed to be copied verbatim to every generation machine

Only one thing needs to change per machine: the `SEED` environment variable (`SB3_GEN_SEED`), or
the `SEED` line in the parameter cell below if setting an env var is inconvenient. Everything else
— output paths, checkpointing, diagnostics — is a pure function of `SEED`, so two machines' outputs
never collide once copied into one shared `data/` folder.

**Not executed.** Run under `caffeinate`/`nohup` per this project's convention for long jobs. Run
the smoke-test cell first and check the printed accept rate before launching a full multi-hour run
on any machine.

## Output files

- `data/wu2003_sbi_train_sb3_widelognormal_seed{SEED}_n{N_TRAIN_PER_SEED}.npz` (+ its
  `.partial.npz` checkpoint sibling)
- `results/22e_sb3_widelognormal_seed{SEED}_sbc_screen_n{N_SBC_LOCAL_SCREEN}.npz` — a small
  per-machine sanity screen, **not** the formal calibration claim (see `notebooks/22f`/`24d` for
  the held-out N=800 confirm check)


## Section 1 — Imports and setup


In [ ]:
import os, pathlib, sys, time, importlib
import numpy as np
import pandas as pd

cwd = pathlib.Path.cwd().resolve()
ROOT = cwd if (cwd / 'src').exists() else cwd.parent
sys.path.insert(0, str(ROOT / 'src'))

import cstr_sbi.recycle.sb3 as sb3
sb3 = importlib.reload(sb3)
from cstr_sbi.recycle.sb3 import (
    LOGN_SIGMA_SB3_4D_WIDE,
    mixed_wide_prior_4d_sb3,
    deterministic_window_sb3,
    noisy_replicates_sb3,
    nominal_warm_start_sb3,
)
from cstr_sbi.recycle.summaries import N_SUMMARIES_SB, compute_summaries

DATA = ROOT / 'data'; DATA.mkdir(exist_ok=True)
RESULTS = ROOT / 'results'; RESULTS.mkdir(exist_ok=True)
LOGS = ROOT / 'logs'; LOGS.mkdir(exist_ok=True)

print(f'ROOT={ROOT}')


## Section 2 — Per-machine parameters

**`SEED` is the only line that should change when this notebook is copied to a new machine** (or set the `SB3_GEN_SEED` environment variable before launching Jupyter).


In [ ]:
SEED = int(os.environ.get('SB3_GEN_SEED', 0))

# Draws to generate on THIS machine. Default 5000; bump per-machine (e.g. up to 8000) so ~4-6
# machines together land in the project's 20,000-50,000 total target. Does NOT need to match
# across machines -- notebooks/22g sums whatever each machine actually produced.
N_TRAIN_PER_SEED = 5000

# Small local sanity screen -- catches gross generation bugs (wrong horizon, non-finite summaries)
# early, without waiting for the full pooled campaign. NOT a calibration gate; the real held-out
# SBC check is notebooks/22f's N=800 confirm set, evaluated once in notebooks/24d.
N_SBC_LOCAL_SCREEN = 60

NOISE_PCT = 0.003
T_FINAL_H = 4.0    # matches notebooks/22b/22c -- required for W11 to pass the settling diagnostic
N_SAVE = 240
CHECKPOINT_EVERY = 20
TIME_CHECKPOINT_S = 60
PROGRESS_EVERY_ATTEMPTS = 5
STEADY_MAX_REL_DRIFT = 0.05
MAX_NORM_CHANNEL = 3.0
MAX_ABS_RAW_OBS = 3.0e7
PRIOR_TAG = 'widelognormal_v1'

PROGRESS_LOG_PATH = LOGS / f'notebook22e_progress_seed{SEED}.log'
TRAIN_PATH = DATA / f'wu2003_sbi_train_sb3_widelognormal_seed{SEED}_n{N_TRAIN_PER_SEED}.npz'
SBC_SCREEN_PATH = RESULTS / f'22e_sb3_widelognormal_seed{SEED}_sbc_screen_n{N_SBC_LOCAL_SCREEN}.npz'

print(f'SEED={SEED}')
print(f'N_TRAIN_PER_SEED={N_TRAIN_PER_SEED}, N_SBC_LOCAL_SCREEN={N_SBC_LOCAL_SCREEN}')
print(f'PRIOR_TAG={PRIOR_TAG}, sigma_eta_col={LOGN_SIGMA_SB3_4D_WIDE[2]:.4f}, '
      f'sigma_xi_reb={LOGN_SIGMA_SB3_4D_WIDE[3]:.4f}')
print(f'TRAIN_PATH={TRAIN_PATH.relative_to(ROOT)}')
print(f'SBC_SCREEN_PATH={SBC_SCREEN_PATH.relative_to(ROOT)}')


## Section 3 — Prior, warm start, RNGs (all seeded from `SEED`)


In [ ]:
import torch
torch.manual_seed(SEED)
rng = np.random.default_rng(SEED)

prior = mixed_wide_prior_4d_sb3(uniform_idx=(0, 1), sigma=LOGN_SIGMA_SB3_4D_WIDE)
y0_sb3 = nominal_warm_start_sb3()
print('Prior and cached nominal S-B3 warm start ready.')


## Section 4 — Generation, diagnostics, and checkpointing helpers

Mirrors `notebooks/22b`/`22c`'s established pattern (deterministic-window simulate, steady-state drift/outlier diagnostics, checkpoint-every-N-accepted-rows) with metadata checks generalized to also guard on `prior_tag`/`seed`, not just horizon.


In [ ]:
def simulate_summary_sb3(theta4, sim_rng, noise_pct=NOISE_PCT, t_final_h=T_FINAL_H, n_save=N_SAVE):
    theta4 = np.asarray(theta4, dtype=np.float32)
    t_h, raw_obs = deterministic_window_sb3(theta4, y0=y0_sb3, t_final_h=t_final_h, n_save=n_save)
    raw_obs = np.asarray(raw_obs)
    noisy = noisy_replicates_sb3(raw_obs, n_replicates=1, rng=sim_rng, noise_pct=noise_pct)[0]
    summary = compute_summaries(noisy, 'S-B', np.asarray(t_h))
    return np.asarray(t_h), raw_obs, noisy, summary


def window_diag(raw_obs):
    raw_obs = np.asarray(raw_obs)
    check_idx = [0, 1, 2, 5, 6, 7, 8, 9, 10]
    norm_idx = [7, 8, 9, 10]
    q_start = max(0, 3 * len(raw_obs) // 4)
    last_q = raw_obs[q_start:]
    final = raw_obs[-1]
    q_mean = last_q.mean(axis=0)
    denom = np.maximum(np.abs(q_mean), 1e-6)
    rel_drift = np.abs(final - q_mean) / denom
    return {
        'max_rel_drift_sb': float(np.max(rel_drift[check_idx])),
        'max_norm_channel': float(np.max(np.abs(raw_obs[:, norm_idx]))),
        'max_abs_raw': float(np.max(np.abs(raw_obs))),
    }


def is_acceptable_window(diag):
    return (
        diag['max_rel_drift_sb'] <= STEADY_MAX_REL_DRIFT
        and diag['max_norm_channel'] <= MAX_NORM_CHANNEL
        and diag['max_abs_raw'] <= MAX_ABS_RAW_OBS
    )


def _check_bank_metadata(data, candidate_name, expected_seed):
    # Guard against resuming from a checkpoint generated under a different prior/seed/horizon
    # -- e.g. a stale narrow-lognormal partial file left at the same path.
    stored_tag = str(data['prior_tag']) if 'prior_tag' in data.files else None
    stored_seed = int(data['seed']) if 'seed' in data.files else None
    stored_t_final_h = float(data['t_final_h']) if 't_final_h' in data.files else None
    stored_n_save = int(data['n_save']) if 'n_save' in data.files else None
    mismatch = (
        (stored_tag is not None and stored_tag != PRIOR_TAG)
        or (stored_seed is not None and stored_seed != expected_seed)
        or (stored_t_final_h is not None and not np.isclose(stored_t_final_h, T_FINAL_H))
        or (stored_n_save is not None and stored_n_save != N_SAVE)
    )
    if mismatch:
        raise ValueError(
            f'{candidate_name} was generated with prior_tag={stored_tag}, seed={stored_seed}, '
            f't_final_h={stored_t_final_h}, n_save={stored_n_save}, which does not match the '
            f'current PRIOR_TAG={PRIOR_TAG}, SEED={expected_seed}, T_FINAL_H={T_FINAL_H}, '
            f'N_SAVE={N_SAVE}. Delete or rename {candidate_name} to regenerate.'
        )


def load_or_init_bank(out_path, expected_seed):
    for candidate in (out_path, out_path.with_suffix('.partial.npz')):
        if candidate.exists():
            data = np.load(candidate)
            _check_bank_metadata(data, candidate.name, expected_seed)
            theta_rows = list(np.asarray(data['thetas4'], dtype=np.float32))
            summary_rows = list(np.asarray(data['summaries'], dtype=np.float32))
            attempts = int(data['attempts']) if 'attempts' in data.files else len(theta_rows)
            return theta_rows, summary_rows, attempts
    return [], [], 0


def save_bank(out_path, theta_rows, summary_rows, seed, attempts):
    theta_arr = np.asarray(theta_rows, dtype=np.float32)
    summary_arr = np.asarray(summary_rows, dtype=np.float32)
    assert summary_arr.shape[1] == N_SUMMARIES_SB
    np.savez(
        out_path,
        thetas4=theta_arr,
        summaries=summary_arr,
        seed=seed,
        attempts=attempts,
        structure='S-B3',
        prior_tag=PRIOR_TAG,
        sigma_eta_col=float(LOGN_SIGMA_SB3_4D_WIDE[2]),
        sigma_xi_reb=float(LOGN_SIGMA_SB3_4D_WIDE[3]),
        t_final_h=T_FINAL_H,
        n_save=N_SAVE,
    )


def _emit_progress(msg):
    line = f'[{time.strftime("%H:%M:%S")}] {msg}'
    print(line)
    with open(PROGRESS_LOG_PATH, 'a') as f:
        f.write(line + '\\n')


def generate_bank(out_path, n_target, seed, sim_rng, checkpoint_every=CHECKPOINT_EVERY,
                   time_checkpoint_s=TIME_CHECKPOINT_S,
                   progress_every_attempts=PROGRESS_EVERY_ATTEMPTS):
    theta_rows, summary_rows, attempts = load_or_init_bank(out_path, seed)
    if len(theta_rows) >= n_target:
        _emit_progress(f'{out_path.name}: already has {len(theta_rows)}/{n_target} rows, done.')
        return (np.asarray(theta_rows, dtype=np.float32)[:n_target],
                np.asarray(summary_rows, dtype=np.float32)[:n_target])

    _emit_progress(f'{out_path.name}: resuming with {len(theta_rows)}/{n_target} accepted rows, '
                    f'{attempts} attempts so far.')
    partial_path = out_path.with_suffix('.partial.npz')
    t_start = time.time()
    t_last_checkpoint = t_start
    reject_nonfinite = 0
    reject_diag = 0

    while len(theta_rows) < n_target:
        theta4 = prior.sample((1,)).numpy()[0]
        attempts += 1
        try:
            _, raw_obs, _, summary = simulate_summary_sb3(theta4, sim_rng)
        except Exception as exc:
            reject_nonfinite += 1
            if attempts % progress_every_attempts == 0:
                _emit_progress(f'attempt {attempts}: solver exception ({exc!r}), rejected.')
            continue

        if not np.isfinite(summary).all():
            reject_nonfinite += 1
            continue
        diag = window_diag(raw_obs)
        if not is_acceptable_window(diag):
            reject_diag += 1
            continue

        theta_rows.append(theta4.astype(np.float32))
        summary_rows.append(summary.astype(np.float32))

        if attempts % progress_every_attempts == 0:
            elapsed = time.time() - t_start
            rate = len(theta_rows) / max(elapsed, 1e-6)
            _emit_progress(
                f'attempt {attempts}: accepted={len(theta_rows)}/{n_target} '
                f'(reject_nonfinite={reject_nonfinite}, reject_diag={reject_diag}), '
                f'{rate:.3f} accepted/s'
            )

        now = time.time()
        if len(theta_rows) % checkpoint_every == 0 or (now - t_last_checkpoint) > time_checkpoint_s:
            save_bank(partial_path, theta_rows, summary_rows, seed, attempts)
            t_last_checkpoint = now
            _emit_progress(f'checkpointed {len(theta_rows)}/{n_target} rows to {partial_path.name}')

    save_bank(out_path, theta_rows, summary_rows, seed, attempts)
    if partial_path.exists():
        partial_path.unlink()
    accept_rate = len(theta_rows) / max(attempts, 1)
    _emit_progress(
        f'DONE {out_path.name}: {len(theta_rows)} rows, {attempts} attempts, '
        f'accept_rate={accept_rate:.3f}, reject_nonfinite={reject_nonfinite}, reject_diag={reject_diag}'
    )
    return np.asarray(theta_rows, dtype=np.float32), np.asarray(summary_rows, dtype=np.float32)


## Section 5 — Smoke test (run this BEFORE the full campaign on any new machine)

Per this project's established discipline before every prior change (see `notebooks/22c` and the various `smoketest_sc2_*.py` scripts): never commit hours of compute to an untested prior. Draws a small number of samples through the real simulator and reports accept rate and per-draw wall time.


In [ ]:
N_SMOKE = 50
_smoke_accept = 0
_t0 = time.time()
for _ in range(N_SMOKE):
    theta4 = prior.sample((1,)).numpy()[0]
    try:
        _, raw_obs, _, summary = simulate_summary_sb3(theta4, rng)
        if np.isfinite(summary).all() and is_acceptable_window(window_diag(raw_obs)):
            _smoke_accept += 1
    except Exception:
        pass
_elapsed = time.time() - _t0
print(f'Smoke test: {_smoke_accept}/{N_SMOKE} accepted ({_smoke_accept / N_SMOKE:.1%}), '
      f'{_elapsed:.1f}s total, {_elapsed / N_SMOKE:.2f}s/attempt.')
print('Compare against notebooks/22b\'s baseline (~91-95% accept, narrow-lognormal mixed-wide '
      'prior). If accept rate has collapsed towards the ~37-47% seen elsewhere in this project for '
      'full-uniform priors, STOP and reconsider before launching the full run.')


## Section 6 — Generate this machine's training chunk

Launch via `jupyter nbconvert --to notebook --execute` under `caffeinate`/`nohup`, not interactively — a full chunk can take hours.


In [ ]:
theta_train, summary_train = generate_bank(TRAIN_PATH, N_TRAIN_PER_SEED, SEED, rng)
print(f'Training chunk: {theta_train.shape}, {summary_train.shape}')


## Section 7 — Local SBC screen (sanity check only)

The formal, adequately-powered calibration check happens once in `notebooks/24d`, against `notebooks/22f`'s dedicated held-out N=800 confirm set (per `finding_sbc_power_eta_col`: N=200 gives false passes). This screen only exists to catch gross per-machine bugs early.


In [ ]:
theta_screen, summary_screen = generate_bank(SBC_SCREEN_PATH, N_SBC_LOCAL_SCREEN, SEED, rng)
print(f'Local SBC screen set: {theta_screen.shape}, {summary_screen.shape}')


## Section 8 — Summary and next steps


In [ ]:
print('=== Section summary ===')
print(f'SEED={SEED}')
print(f'Training chunk: {TRAIN_PATH.name} -> {theta_train.shape[0]} rows')
print(f'Local SBC screen: {SBC_SCREEN_PATH.name} -> {theta_screen.shape[0]} rows')
print()
print('Next steps:')
print('  1. Copy this notebook + src/ to the next machine, set a different SB3_GEN_SEED, repeat.')
print('  2. Once all machines are done, run notebooks/22f ONCE (centrally) for the held-out N=800 '
      'confirm set.')
print("  3. Copy every machine's data/wu2003_sbi_train_sb3_widelognormal_seed*_n*.npz into one "
      'shared data/ folder and run notebooks/22g to pool them.')
print('  4. Run notebooks/24d to train, run the held-out SBC check, and classify.')
